# 03 - Modélisation (section 2.2)



In [ ]:
import sys
from pathlib import Path

# Remonte jusqu'à la racine du projet (le dossier qui contient src/), quel que soit le dossier de lancement
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

from src.load_data import load_wine_data, make_binary_target, split_data
from src.preprocess import clean_data
from src.models import (
    LOGREG_GRID, TREE_GRID, decision_tree, load_selected_features, logistic_regression, save_model,
)
from src.tuning import run_grid_search
from src.evaluate import train_vs_cv_scores

## 1. Données et variables retenues

In [ ]:
df = make_binary_target(clean_data(load_wine_data()))
X_train, X_test, y_train, y_test = split_data(df)
features = load_selected_features()

print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Variables retenues (notebook 02) :", features)

## 2. Régression logistique

### 2.1 Modèle par défaut (C = 1, L2)

In [ ]:
logreg_base = logistic_regression(features)
pd.Series(train_vs_cv_scores(logreg_base, X_train, y_train)).round(3)

### 2.2 Grid Search sur `C` et la pénalité

In [ ]:
logreg_search = run_grid_search(logistic_regression(features), LOGREG_GRID, X_train, y_train)

print("Meilleurs hyperparamètres :", logreg_search.best_params_)
print("F1 moyen en validation croisée :", round(logreg_search.best_score_, 3))

In [ ]:
logreg_cv = pd.DataFrame(logreg_search.cv_results_)
pivot = logreg_cv.pivot_table(index="param_model__C", columns="param_model__l1_ratio", values="mean_test_score")
pivot.index = [f"{c:.3g}" for c in pivot.index]

plt.figure(figsize=(6, 6))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis")
plt.xlabel("l1_ratio (0 = L2, 0.5 = elastic-net, 1 = L1)")
plt.ylabel("C")
plt.title("Régression logistique : F1 moyen en validation croisée")
plt.tight_layout()
plt.show()

In [ ]:
coefs = pd.Series(
    logreg_search.best_estimator_.named_steps["model"].coef_[0], index=features
).sort_values()

coefs.plot.barh(figsize=(7, 3), color=["tab:red" if c < 0 else "tab:green" for c in coefs])
plt.axvline(0, color="black", lw=0.8)
plt.title("Coefficients de la régression logistique (variables standardisées)")
plt.tight_layout()
plt.show()
coefs.round(3)

## 3. Arbre de décision

### 3.1 Arbre non contraint : mise en évidence du surapprentissage

In [ ]:
tree_base = decision_tree(features)
pd.Series(train_vs_cv_scores(tree_base, X_train, y_train)).round(3)

**Analyse.** F1 = 1,00 sur le train mais 0,39 en validation : l'arbre non contraint apprend le train par cœur, bruit compris. C'est l'illustration typique d'une **variance élevée**, et il fait moins bien que la régression logistique (0,525).

### 3.2 Grid Search sur `max_depth` et `min_samples_leaf`

In [ ]:
tree_search = run_grid_search(decision_tree(features), TREE_GRID, X_train, y_train)

print("Meilleurs hyperparamètres :", tree_search.best_params_)
print("F1 moyen en validation croisée :", round(tree_search.best_score_, 3))

In [ ]:
tree_cv = pd.DataFrame(tree_search.cv_results_)
tree_cv["param_model__max_depth"] = tree_cv["param_model__max_depth"].fillna("None").astype(str)
depth_order = [str(d) for d in TREE_GRID["model__max_depth"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, score, title in [(axes[0], "mean_train_score", "Entraînement"), (axes[1], "mean_test_score", "Validation croisée")]:
    pivot = tree_cv.pivot_table(index="param_model__max_depth", columns="param_model__min_samples_leaf", values=score)
    sns.heatmap(pivot.loc[depth_order], annot=True, fmt=".2f", cmap="viridis", ax=ax, vmin=0.3, vmax=1)
    ax.set_xlabel("min_samples_leaf")
    ax.set_ylabel("max_depth")
    ax.set_title(f"Arbre : F1 moyen ({title})")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.tree import plot_tree

best_tree = tree_search.best_estimator_.named_steps["model"]

plt.figure(figsize=(18, 7))
plot_tree(best_tree, feature_names=features, class_names=["moins bon", "bon"],
          filled=True, rounded=True, max_depth=3, fontsize=9)
plt.title("Arbre optimisé (3 premiers niveaux, seuils sur variables standardisées)")
plt.show()

pd.Series(best_tree.feature_importances_, index=features).sort_values(ascending=False).round(3)

## 4. Sauvegarde des modèles optimisés

`GridSearchCV` réentraîne le meilleur pipeline sur tout le train (`refit=True`). Le test n'a pas encore été utilisé.

In [ ]:
save_model(logreg_search.best_estimator_, "regression_logistique")
save_model(tree_search.best_estimator_, "arbre_decision")
print(sorted(p.name for p in (ROOT / "models").glob("*.joblib")))